In [1]:
# Se estiver em ambiente limpo/virtualenv:
!pip -q install "transformers>=4.42.0" "adapters>=1.2.0" torch --upgrade

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.6.0+cu118 requires torch==2.6.0+cu118, but you have torch 2.8.0 which is incompatible.
torchvision 0.21.0+cu118 requires torch==2.6.0+cu118, but you have torch 2.8.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"  # impede import de torchvision


In [ ]:
import torch
from transformers import AutoTokenizer
from adapters import AutoAdapterModel

In [5]:
from typing import List, Tuple

def collapse_bio(lbl: str) -> str:
    return "B" if lbl.startswith("B-") else ("I" if lbl.startswith("I-") else "O")

def bio_to_spans(tokens: List[str], bio_tags: List[str]) -> List[Tuple[int, int, str]]:
    spans, start = [], None
    for i, tag in enumerate(bio_tags):
        if tag == "B":
            if start is not None:
                spans.append((start, i, " ".join(tokens[start:i])))
            start = i
        elif tag == "I":
            if start is None:
                start = i
        else:
            if start is not None:
                spans.append((start, i, " ".join(tokens[start:i])))
                start = None
    if start is not None:
        spans.append((start, len(tokens), " ".join(tokens[start:])))
    return spans

In [6]:
# Carrega RoBERTa + adapter de CHUNKING (CoNLL-2000)
# >>> CORREÇÃO AQUI: add_prefix_space=True <<<
tokenizer = AutoTokenizer.from_pretrained(
    "roberta-base",
    use_fast=True,
    add_prefix_space=True,   # necessário para entradas pré-tokenizadas
)

model = AutoAdapterModel.from_pretrained("roberta-base")
adapter_name = model.load_adapter("AdapterHub/roberta-base-pf-conll2000", source="hf")
model.set_active_adapters(adapter_name)
model.eval()

id2label = model.get_labels_dict()  # mapeia ids -> rótulos (O, B-NP, I-NP, B-VP, ...)

# Mapa id<->label da head ativa (ex.: O, B-NP, I-NP, B-VP, ...)
id2label = model.get_labels_dict()    # AdapterHub expõe esse dict

Some weights of RobertaAdapterModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['heads.default.3.bias', 'roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


In [8]:
def tag_sentence(text: str):
    # pré-tokeniza por espaço para alinhar token↔palavra
    words = text.split()

    # RoBERTa requer add_prefix_space=True quando usamos is_split_into_words
    enc = tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True)

    with torch.no_grad():
        logits = model(**enc).logits  # [1, seq_len, num_labels]
    pred_ids = logits.argmax(-1)[0].tolist()
    word_ids = enc.word_ids(0)  # mapa subword->palavra

    seen, words_out, full_labels = set(), [], []
    for i, wid in enumerate(word_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        words_out.append(words[wid])
        full_labels.append(id2label[pred_ids[i]])

    bio = [collapse_bio(lbl) for lbl in full_labels]
    spans = bio_to_spans(words_out, bio)
    return words_out, full_labels, bio, spans

In [9]:
for s in [
    "Book a table for two at 7 pm in Soho.",
    "Schedule a meeting with Anna tomorrow morning.",
    "Play the latest album by Daft Punk."
]:
    w, full, bio, spans = tag_sentence(s)
    print("\n▸", s)
    print("Words:", w)
    print("Tags :", full)  # B-NP / I-VP / O ...
    print("BIO  :", bio)   # B / I / O
    print("Spans:", spans) # (start, end, 'texto')


▸ Book a table for two at 7 pm in Soho.
Words: ['Book', 'a', 'table', 'for', 'two', 'at', '7', 'pm', 'in', 'Soho.']
Tags : ['B-VP', 'B-NP', 'I-NP', 'B-PP', 'B-NP', 'B-PP', 'B-NP', 'I-NP', 'B-PP', 'B-NP']
BIO  : ['B', 'B', 'I', 'B', 'B', 'B', 'B', 'I', 'B', 'B']
Spans: [(0, 1, 'Book'), (1, 3, 'a table'), (3, 4, 'for'), (4, 5, 'two'), (5, 6, 'at'), (6, 8, '7 pm'), (8, 9, 'in'), (9, 10, 'Soho.')]

▸ Schedule a meeting with Anna tomorrow morning.
Words: ['Schedule', 'a', 'meeting', 'with', 'Anna', 'tomorrow', 'morning.']
Tags : ['B-VP', 'B-NP', 'I-NP', 'B-PP', 'B-NP', 'B-NP', 'I-NP']
BIO  : ['B', 'B', 'I', 'B', 'B', 'B', 'I']
Spans: [(0, 1, 'Schedule'), (1, 3, 'a meeting'), (3, 4, 'with'), (4, 5, 'Anna'), (5, 7, 'tomorrow morning.')]

▸ Play the latest album by Daft Punk.
Words: ['Play', 'the', 'latest', 'album', 'by', 'Daft', 'Punk.']
Tags : ['B-PP', 'B-NP', 'I-NP', 'I-NP', 'B-PP', 'B-NP', 'I-NP']
BIO  : ['B', 'B', 'I', 'I', 'B', 'B', 'I']
Spans: [(0, 1, 'Play'), (1, 4, 'the latest album